# TreePartitioner: Automatic Subgroup Discovery for Conditional PFI

This notebook demonstrates how to use `TreePartitioner` for automatic subgroup discovery in Conditional Subgroup Permutation Feature Importance (cs-PFI).

## What is cs-PFI?

Standard Permutation Feature Importance (PFI) can produce misleading results when features are correlated. cs-PFI addresses this by permuting feature values **only within homogeneous subgroups** where feature correlations are preserved.

## TreePartitioner vs ManualPartitioner

| Partitioner | When to Use |
|-------------|-------------|
| `TreePartitioner` | When you don't know natural groupings - learns subgroups automatically from data |
| `ManualPartitioner` | When domain knowledge suggests groupings (e.g., regions, product categories) |

**Note:** This notebook requires the skforecast dependency:
```bash
pip install xeries[skforecast]
```

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.tree import plot_tree

from skforecast.recursive import ForecasterRecursiveMultiSeries
from xeries import ConditionalPermutationImportance, TreePartitioner
from xeries.adapters.skforecast import from_skforecast

## 1. Generate Synthetic Data with Exogenous Features

We create 4 series (2 urban, 2 rural) with **heterogeneous feature effects**:

### Data Generating Process (DGP)

**Urban Series (MT_001, MT_002):**
```
y[t] = base + 0.5 * y[t-1] + 0.8 * (temperature - 15) + 15 * promotion + noise
```

**Rural Series (MT_003, MT_004):**
```
y[t] = base + 0.7 * y[t-1] + 0.1 * (temperature - 15) + 25 * promotion + noise
```

### Ground Truth Coefficients

| Feature | Urban | Rural |
|---------|-------|-------|
| lag_1 | 0.5 | 0.7 |
| temperature | 0.8 (strong) | 0.1 (weak) |
| promotion | 15 | 25 |

In [8]:
np.random.seed(42)
n_periods = 400
dates = pd.date_range('2023-01-01', periods=n_periods, freq='D')

series_config = {
    'MT_001': {'type': 'urban', 'base': 100, 'ar_coef': 0.5, 'temp_coef': 0.8, 'promo_coef': 15},
    'MT_002': {'type': 'urban', 'base': 120, 'ar_coef': 0.5, 'temp_coef': 0.8, 'promo_coef': 15},
    'MT_003': {'type': 'rural', 'base': 80, 'ar_coef': 0.7, 'temp_coef': 0.1, 'promo_coef': 25},
    'MT_004': {'type': 'rural', 'base': 90, 'ar_coef': 0.7, 'temp_coef': 0.1, 'promo_coef': 25},
}

series_data = {}
exog_data = {}

for series_id, config in series_config.items():
    np.random.seed(hash(series_id) % 2**32)
    
    temperature = 15 + 10 * np.sin(2 * np.pi * np.arange(n_periods) / 365) + np.random.randn(n_periods) * 2
    promotion = (np.random.rand(n_periods) < 0.2).astype(float)
    
    values = np.zeros(n_periods)
    values[0] = config['base']
    
    for t in range(1, n_periods):
        ar_effect = config['ar_coef'] * values[t-1]
        temp_effect = config['temp_coef'] * (temperature[t] - 15)
        promo_effect = config['promo_coef'] * promotion[t]
        noise = np.random.randn() * 3
        
        values[t] = config['base'] * 0.3 + ar_effect + temp_effect + promo_effect + noise
    
    series_data[series_id] = values
    exog_data[series_id] = pd.DataFrame({
        'temperature': temperature,
        'promotion': promotion
    }, index=dates)

df = pd.DataFrame(series_data, index=dates)
print(f"Series shape: {df.shape}")
print(f"Series IDs: {list(df.columns)}")
df.head()

Series shape: (400, 4)
Series IDs: ['MT_001', 'MT_002', 'MT_003', 'MT_004']


,MT_001,MT_002,MT_003,MT_004
2023-01-01,100.000000,120.000000,80.000000,90.000000
2023-01-02,77.295831,118.660016,80.656575,92.083560
2023-01-03,80.948052,106.345940,79.423876,89.838453
2023-01-04,71.296375,89.143501,75.491744,95.094885
2023-01-05,69.274747,82.665728,99.766660,118.444362


## 2. Prepare Exogenous Data

skforecast expects exogenous features as a dictionary of DataFrames (one per series).

In [9]:
print("Exogenous features per series:")
for series_id, exog_df in exog_data.items():
    print(f"  {series_id}: {list(exog_df.columns)}")

print(f"\nExogenous data for MT_001:")
display(exog_data['MT_001'].head())

Exogenous features per series:
  MT_001: ['temperature', 'promotion']
  MT_002: ['temperature', 'promotion']
  MT_003: ['temperature', 'promotion']
  MT_004: ['temperature', 'promotion']

Exogenous data for MT_001:


,temperature,promotion
2023-01-01,13.458184,1.0
2023-01-02,15.708320,0.0
2023-01-03,14.017892,1.0
2023-01-04,17.310672,0.0
2023-01-05,15.115266,0.0


## 3. Train ForecasterRecursiveMultiSeries

We use skforecast's `ForecasterRecursiveMultiSeries` which automatically creates lag features and handles exogenous variables.

We configure `transformer_series` and `transformer_exog` to standardize features internally, which ensures all features contribute equally to TreePartitioner's decision tree splits.

In [ ]:
forecaster = ForecasterRecursiveMultiSeries(
    estimator=RandomForestRegressor(
        n_estimators=50,
        max_depth=10,
        random_state=42
    ),
    lags=3,
    transformer_series=StandardScaler(),
    transformer_exog=StandardScaler(),
)

forecaster.fit(series=df, exog=exog_data)
print(f"Forecaster fitted with lags: {forecaster.lags}")


╭────────────────────────────────── InputTypeWarning ──────────────────────────────────╮
│ Passing a DataFrame (either wide or long format) as `series` requires additional     │
│ internal transformations, which can increase computational time. It is recommended   │
│ to use a dictionary of pandas Series instead. For more details, see:                 │
│ https://skforecast.org/latest/user_guides/independent-multi-time-series-forecasting. │
│ html#input-data                                                                      │
│                                                                                      │
│ Category : skforecast.exceptions.InputTypeWarning                                    │
│ Location :                                                                           │
│ c:\Workspace\Research\time_conditional_pfi\.venv\Lib\site-packages\skforecast\utils\ │
│ utils.py:2568                                                                        │
│ Suppress : warnings.simplefilter('ignore', category=InputTypeWarning)                │
╰──────────────────────────────────────────────────────────────────────────────────────╯

Forecaster fitted with lags: [1 2 3]


AttributeError: 'ForecasterRecursiveMultiSeries' object has no attribute 'exog_col_names'

## 4. Extract Training Data with Adapter

The `from_skforecast` adapter extracts the training matrix (X, y) in the format expected by xeries.

In [ ]:
adapter = from_skforecast(forecaster, series=df, exog=exog_data)
X, y = adapter.get_training_data()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nFeatures: {list(X.columns)}")
display(X.head())

╭────────────────────────────────── InputTypeWarning ──────────────────────────────────╮
│ Passing a DataFrame (either wide or long format) as `series` requires additional     │
│ internal transformations, which can increase computational time. It is recommended   │
│ to use a dictionary of pandas Series instead. For more details, see:                 │
│ https://skforecast.org/latest/user_guides/independent-multi-time-series-forecasting. │
│ html#input-data                                                                      │
│                                                                                      │
│ Category : skforecast.exceptions.InputTypeWarning                                    │
│ Location :                                                                           │
│ c:\Workspace\Research\time_conditional_pfi\.venv\Lib\site-packages\skforecast\utils\ │
│ utils.py:2568                                                                        │
│ Suppress : warnings.simplefilter('ignore', category=InputTypeWarning)                │
╰──────────────────────────────────────────────────────────────────────────────────────╯

X shape: (1588, 6)
y shape: (1588,)

Features: ['lag_1', 'lag_2', 'lag_3', '_level_skforecast', 'temperature', 'promotion']


,lag_1,lag_2,lag_3,_level_skforecast,temperature,promotion
2023-01-04,80.948052,77.295831,100.000000,0,17.310672,0.0
2023-01-05,71.296375,80.948052,77.295831,0,15.115266,0.0
2023-01-06,69.274747,71.296375,80.948052,0,12.684350,0.0
2023-01-07,58.787725,69.274747,71.296375,0,15.771915,1.0
2023-01-08,70.220812,58.787725,69.274747,0,15.867181,0.0


## 5. Data Standardization (Handled by skforecast)

Features have different scales:
- `temperature`: ~5 to 25 (continuous)
- `promotion`: 0 or 1 (binary)
- `lag_*`: varies based on series values

Standardization ensures all features contribute equally to the decision tree splits in TreePartitioner.

**Note:** We configured `transformer_series=StandardScaler()` and `transformer_exog=StandardScaler()` in the forecaster, so the training data extracted via the adapter is already standardized.

In [ ]:
feature_cols = [c for c in X.columns if c != '_level_skforecast']

print("Feature statistics (standardized by skforecast):")
print(X[feature_cols].describe().round(2))

## 6. TreePartitioner Basics

TreePartitioner implements the cs-PFI algorithm:

1. **Trains a decision tree** to predict the feature being permuted using all other features
2. **Uses leaf nodes as subgroups** - samples in the same leaf have similar feature values
3. **Permutes within groups** - preserves feature correlations during importance calculation

### Key Parameters

| Parameter | Description | Default |
|-----------|-------------|---------|
| `max_depth` | Maximum tree depth | 4 |
| `min_samples_leaf` | Minimum samples per leaf (int or fraction) | 0.05 |
| `series_col` | Column with series identifiers | Auto-detects |
| `random_state` | Random seed | None |

In [ ]:
partitioner = TreePartitioner(
    max_depth=4,
    min_samples_leaf=0.05,
    random_state=42
)

partitioner.fit(X, feature='lag_1')

print(f"Number of groups (leaf nodes): {partitioner.n_groups}")
print(f"Tree depth: {partitioner.tree.get_depth()}")

groups = partitioner.get_groups(X)
print(f"\nGroup assignments shape: {groups.shape}")
print(f"Unique groups: {np.unique(groups)}")
print(f"\nSamples per group:")
for g in np.unique(groups):
    print(f"  Group {g}: {np.sum(groups == g)} samples")

## 7. Visualize Tree Structure

The decision tree shows how samples are partitioned into subgroups based on feature values.

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))

tree_feature_names = [c for c in X.columns if c not in ['lag_1', '_level_skforecast', 'date']]

plot_tree(
    partitioner.tree,
    feature_names=tree_feature_names,
    filled=True,
    rounded=True,
    ax=ax,
    fontsize=10
)
ax.set_title('TreePartitioner: Decision Tree for lag_1 Partitioning', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Use with ConditionalPermutationImportance

Now we use the TreePartitioner with `ConditionalPermutationImportance` to compute feature importance.

In [ ]:
partitioner = TreePartitioner(
    max_depth=4,
    min_samples_leaf=0.05,
    random_state=42
)

explainer = ConditionalPermutationImportance(
    model=adapter,
    metric='mse',
    partitioner=partitioner,
    n_repeats=5,
    n_jobs=1,
    random_state=42
)

result = explainer.explain(
    X=X,
    y=y,
    features=['lag_1', 'lag_2', 'lag_3', 'temperature', 'promotion']
)

print("Feature Importance Results:")
display(result.to_dataframe())

## 9. Visualize Results

In [ ]:
from xeries.visualization import plot_importance_bar

fig, ax = plot_importance_bar(
    result,
    title='Conditional Permutation Feature Importance (TreePartitioner)',
    figsize=(10, 5)
)
plt.tight_layout()
plt.show()

## 10. Parameter Tuning

The `max_depth` and `min_samples_leaf` parameters control the granularity of subgroups:

- **More groups** (deeper tree, smaller leaves): finer-grained conditioning, but may have too few samples per group
- **Fewer groups** (shallower tree, larger leaves): more robust estimates, but may not fully capture correlations

In [ ]:
print("Effect of TreePartitioner parameters on number of groups:")
print("="*60)
print(f"{'max_depth':>12} | {'min_samples_leaf':>18} | {'n_groups':>10}")
print("-"*60)

param_combinations = [
    (2, 0.10),
    (3, 0.10),
    (4, 0.10),
    (4, 0.05),
    (4, 0.02),
    (5, 0.05),
    (6, 0.02),
]

for max_depth, min_samples_leaf in param_combinations:
    p = TreePartitioner(
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )
    p.fit(X, feature='lag_1')
    print(f"{max_depth:>12} | {min_samples_leaf:>18} | {p.n_groups:>10}")

print("="*60)
print("\nTip: Start with defaults (max_depth=4, min_samples_leaf=0.05)")
print("     and adjust based on your dataset size and correlation structure.")

## 11. Summary

### Key Takeaways

| Aspect | Details |
|--------|--------|
| **What TreePartitioner does** | Automatically discovers homogeneous subgroups using decision tree leaf nodes |
| **When to use** | When you don't have predefined groupings and want data-driven subgroup discovery |
| **Key parameters** | `max_depth` (tree depth), `min_samples_leaf` (minimum samples per group) |
| **Best practice** | Standardize features to ensure equal contribution to tree splits (use `transformer_series` and `transformer_exog` in skforecast) |
| **Alternative** | Use `ManualPartitioner` when domain knowledge suggests natural groupings |

### Comparison with strategy='auto'

When you use `ConditionalPermutationImportance(strategy='auto')`, it internally creates a `TreePartitioner` with default parameters. Explicitly creating a `TreePartitioner` allows you to:

1. **Tune parameters** (`max_depth`, `min_samples_leaf`) for your specific dataset
2. **Inspect the tree** structure to understand how subgroups are formed
3. **Reuse the same partitioner** across multiple feature importance calculations